# Structure-to-Structure Backtest
## Real Data + Monte Carlo Simulation

Buy at H1 support, sell at H1 resistance, target next structure level.

In [ ]:
# 1. Setup: mount Drive, load data
import os, time
import numpy as np
import pandas as pd
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/mt5_ea/colab_export'
print('Data files:', len([f for f in os.listdir(DATA_DIR) if f.endswith('.parquet')]))

In [ ]:
# 2. Load data and find structure levels
SYMBOLS = ['EURUSD', 'GBPUSD', 'AUDUSD', 'NZDUSD']
PIP_VAL = {'EURUSD': 0.0001, 'GBPUSD': 0.0001, 'AUDUSD': 0.0001, 'NZDUSD': 0.0001}

def load_data(sym, tf):
    p = f'{DATA_DIR}/{sym}_{tf}.parquet'
    if not os.path.exists(p):
        return None
    df = pd.read_parquet(p)
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
    return df

def find_swings(df, lookback=20):
    """Vectorized swing detection."""
    highs = df['high'].values
    lows = df['low'].values
    n = len(highs)
    swing_highs = []
    swing_lows = []
    for i in range(lookback, n - lookback):
        if highs[i] == np.max(highs[i-lookback:i+lookback+1]):
            swing_highs.append((i, highs[i]))
        if lows[i] == np.min(lows[i-lookback:i+lookback+1]):
            swing_lows.append((i, lows[i]))
    return swing_highs, swing_lows

# Load and analyze
for sym in SYMBOLS:
    df_h1 = load_data(sym, 'H1')
    df_m15 = load_data(sym, 'M15')
    if df_h1 is None or df_m15 is None:
        print(f'{sym}: NO DATA')
        continue
    sh, sl = find_swings(df_h1, 20)
    print(f'{sym}: H1={len(df_h1)} bars, M15={len(df_m15)} bars, SwingHighs={len(sh)}, SwingLows={len(sl)}')

In [ ]:
# 3. Structure-to-Structure backtest engine
def backtest_structure(df_m15, df_h1, pip, stop_pips=20):
    """Backtest: buy at support, sell at resistance, target next structure."""
    swing_highs, swing_lows = find_swings(df_h1, 20)
    m15_times = df_m15['time'].values
    trades = []
    used_levels = set()
    
    # Buy at support
    for idx, price in swing_lows:
        if idx >= len(df_h1): continue
        level_time = df_h1['time'].iloc[idx]
        
        # Find next resistance
        next_res = None
        for r_idx, r_price in swing_highs:
            if r_idx > idx and r_price > price:
                next_res = r_price
                break
        if next_res is None: continue
        
        target_pips = (next_res - price) / pip
        if target_pips < stop_pips * 1.5: continue  # min 1.5:1 R:R
        
        level_idx = np.searchsorted(m15_times, level_time)
        level_key = ('S', round(price, 5))
        if level_key in used_levels: continue
        
        for j in range(level_idx, min(level_idx + 200, len(df_m15))):
            row = df_m15.iloc[j]
            if abs(row['low'] - price) < 5 * pip and row['close'] > row['open']:
                used_levels.add(level_key)
                for k in range(j + 1, min(j + 200, len(df_m15))):
                    if df_m15.iloc[k]['low'] <= price - stop_pips * pip:
                        trades.append({'pips': -stop_pips, 'target': target_pips, 'result': 'loss'})
                        break
                    if df_m15.iloc[k]['high'] >= price + target_pips * pip:
                        trades.append({'pips': target_pips, 'target': target_pips, 'result': 'win'})
                        break
                break
    
    # Sell at resistance
    for idx, price in swing_highs:
        if idx >= len(df_h1): continue
        level_time = df_h1['time'].iloc[idx]
        
        next_sup = None
        for s_idx, s_price in swing_lows:
            if s_idx > idx and s_price < price:
                next_sup = s_price
                break
        if next_sup is None: continue
        
        target_pips = (price - next_sup) / pip
        if target_pips < stop_pips * 1.5: continue
        
        level_idx = np.searchsorted(m15_times, level_time)
        level_key = ('R', round(price, 5))
        if level_key in used_levels: continue
        
        for j in range(level_idx, min(level_idx + 200, len(df_m15))):
            row = df_m15.iloc[j]
            if abs(row['high'] - price) < 5 * pip and row['close'] < row['open']:
                used_levels.add(level_key)
                for k in range(j + 1, min(j + 200, len(df_m15))):
                    if df_m15.iloc[k]['high'] >= price + stop_pips * pip:
                        trades.append({'pips': -stop_pips, 'target': target_pips, 'result': 'loss'})
                        break
                    if df_m15.iloc[k]['low'] <= price - target_pips * pip:
                        trades.append({'pips': target_pips, 'target': target_pips, 'result': 'win'})
                        break
                break
    
    return trades

print('Engine ready')

In [ ]:
# 4. Run backtest for all symbols
all_trades = {}

for sym in SYMBOLS:
    df_h1 = load_data(sym, 'H1')
    df_m15 = load_data(sym, 'M15')
    if df_h1 is None or df_m15 is None:
        continue
    
    trades = backtest_structure(df_m15, df_h1, PIP_VAL[sym], stop_pips=20)
    all_trades[sym] = trades
    
    if trades:
        wins = sum(1 for t in trades if t['result'] == 'win')
        total = len(trades)
        wr = wins / total * 100
        avg_win = np.mean([t['pips'] for t in trades if t['result'] == 'win']) if wins else 0
        avg_loss = np.mean([t['pips'] for t in trades if t['result'] == 'loss']) if (total - wins) else 0
        total_pips = sum(t['pips'] for t in trades)
        expectancy = np.mean([t['pips'] for t in trades])
        pf = sum(t['pips'] for t in trades if t['result'] == 'win') / abs(sum(t['pips'] for t in trades if t['result'] == 'loss')) if any(t['result'] == 'loss' for t in trades) else float('inf')
        
        print(f'{sym}: {total} trades | WR: {wr:.1f}% | Avg Win: +{avg_win:.1f} | Avg Loss: -{abs(avg_loss):.1f} | R:R 1:{avg_win/abs(avg_loss):.1f} | EV: {expectancy:+.1f}p | PF: {pf:.2f} | Total: {total_pips:+.1f}p')
    else:
        print(f'{sym}: 0 trades')

In [ ]:
# 5. Monte Carlo with real trade distribution
SIMS = 10000
SIM_DAYS = 330
RISK = 75.0
STARTING = 9526.78
MAX_LOSS_FLOOR = 9000.0
PHASE1 = 10800.0
PHASE2 = 10500.0

# Combine all trades
combined_pnls = []
for sym, trades in all_trades.items():
    for t in trades:
        combined_pnls.append(t['pips'])

combined_pnls = np.array(combined_pnls)
print(f'Combined trades: {len(combined_pnls)}')
print(f'Win Rate: {(combined_pnls > 0).mean()*100:.1f}%')
print(f'Avg Win: +{combined_pnls[combined_pnls > 0].mean():.1f} pips')
print(f'Avg Loss: {combined_pnls[combined_pnls <= 0].mean():.1f} pips')
print(f'Expectancy: {combined_pnls.mean():+.1f} pips/trade')

# Calculate trades per day from data
df_m15 = load_data('EURUSD', 'M15')
if df_m15 is not None:
    total_days = (df_m15['time'].iloc[-1] - df_m15['time'].iloc[0]).days
    trades_per_day = len(combined_pnls) / total_days / len(SYMBOLS)
    print(f'Trades per day: {trades_per_day:.3f}')

In [ ]:
# 6. Run Monte Carlo
t0 = time.time()

results = []
for _ in range(SIMS):
    bal = STARTING
    p1 = False
    p2 = False
    br = False
    n_trades = 0
    
    for day in range(1, SIM_DAYS + 1):
        if np.random.random() < trades_per_day:
            pnl_pips = np.random.choice(combined_pnls)
            pnl_dollars = pnl_pips * (RISK / 20.0)  # $75 risk / 20 pip stop = $3.75/pip
            bal += pnl_dollars
            n_trades += 1
            
            if bal <= MAX_LOSS_FLOOR:
                br = True
                break
            if not p1 and bal >= PHASE1:
                p1 = True
            if p1 and not p2 and bal >= PHASE2:
                p2 = True
                break
    
    results.append({'bal': bal, 'p1': p1, 'p2': p2, 'br': br, 'trades': n_trades, 'days': day})

elapsed = time.time() - t0
print(f'Monte Carlo done: {SIMS:,} sims in {elapsed:.1f}s')

In [ ]:
# 7. Results
total = len(results)
p1_rate = sum(1 for r in results if r['p1']) / total * 100
p2_rate = sum(1 for r in results if r['p2']) / total * 100
br_rate = sum(1 for r in results if r['br']) / total * 100
avg_bal = np.mean([r['bal'] for r in results])
med_bal = np.median([r['bal'] for r in results])
p5 = np.percentile([r['bal'] for r in results], 5)
p95 = np.percentile([r['bal'] for r in results], 95)
avg_days = np.mean([r['days'] for r in results if r['p2']]) if any(r['p2'] for r in results) else 0
avg_trades = np.mean([r['trades'] for r in results])

print('=' * 60)
print('STRUCTURE-TO-STRUCTURE: MONTE CARLO RESULTS')
print('=' * 60)
print(f'Risk per trade: ${RISK:.0f} | Stop: 20 pips | Target: Next structure')
print()
print(f'Pass Phase 1:  {p1_rate:.1f}%')
print(f'Pass Phase 2:  {p2_rate:.1f}%')
print(f'Breach:        {br_rate:.1f}%')
print()
print(f'Avg balance:   ${avg_bal:,.2f}')
print(f'Median balance: ${med_bal:,.2f}')
print(f'5th percentile: ${p5:,.2f}')
print(f'95th percentile: ${p95:,.2f}')
print(f'Avg days to pass: {avg_days:.0f} ({avg_days/30:.1f} months)')
print(f'Avg trades: {avg_trades:.0f}')

# Compare with ATR-based
print()
print('=' * 60)
print('COMPARISON: STRUCTURE vs ATR-BASED (current)')
print('=' * 60)
print(f'STRUCTURE: Pass {p2_rate:.1f}% | Breach {br_rate:.1f}% | Avg ${avg_bal:,.0f} | {avg_days:.0f} days')
print(f'ATR-BASED: Pass 95.9% | Breach 3.4% | Avg $10,754 | 102 days')